<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Phase31_Drive_to_Drive_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICHI 2027 — Phase 31
## Frozen Athena intake, source-semantics audit, and synthetic structural shadow tests
**Google Drive → Google Drive · no fabricated concepts · no private vocabulary in public outputs**

**Inputs:** the verified Phase 30 `SHAREABLE.json`, `BINDING_VALIDATION_STATUS.csv`, `LINEAGE_INTEGRITY_TESTS.csv`, plus the exact Phase 29 shareable JSON, FHIR/Procedure bundle, lineage sidecar, gap disposition and binding audit. Matching runs are selected by the Phase 30 SHA-256 references, not arbitrary filenames.

**What happens without Athena:** independently repeat structural/cardinality/idempotence tests on a vocabulary-free **synthetic shadow representation**, create a 19-row private reviewer template and a private Athena snapshot manifest template, and export public-safe status and gap tables. Missing concepts remain blocked.

**What happens if private Athena files are present:** freeze the local `CONCEPT.csv` by SHA-256, generate a **private suggestion-only candidate list**, validate human-reviewed choices against vocabulary domain, standard status, date validity, and pinned event-field evidence. No concept ID is automatically selected. Even a completed review is **not a production/clinical certification**.

**Scientific boundaries:** Synthetic-only input. Arm A target ETL and OMOP/MI-CDM loading, full cohort equivalence, real ADNI evaluation and four-arm comparisons remain **unexecuted**. Arm C is nonportable as-is. No synthetic sex or race is silently invented for the current source patients.

In [1]:
# 1. Mount Google Drive (local test override is never needed in Colab).
import os, re, json, csv, gzip, hashlib, shutil
from pathlib import Path
from datetime import datetime, timezone, date
import pandas as pd

if os.environ.get('ICHI2027_DRIVE_ROOT'):
    DRIVE_ROOT=Path(os.environ['ICHI2027_DRIVE_ROOT'])
else:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT=Path('/content/drive/MyDrive')

# Optional: e.g. 'NeuroFHIR_ICHI2027/phase24_outputs' if multiple copies exist.
DRIVE_RELATIVE_FOLDER = ''
ROOT = DRIVE_ROOT / DRIVE_RELATIVE_FOLDER if DRIVE_RELATIVE_FOLDER else DRIVE_ROOT
assert ROOT.is_dir(), f'Drive folder not available: {ROOT}'
print('Drive search folder:', ROOT)

Mounted at /content/drive
Drive search folder: /content/drive/MyDrive


In [2]:
# 2. Stale-safe discovery and exact Phase 29/30 parent-run matching.
def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1<<20), b''):h.update(b)
    return h.hexdigest()

def find_files(root, pattern):
    hits=[]
    for directory, children, names in os.walk(root,onerror=lambda _:None):
        children[:]=[n for n in children if n not in {'_PRIVATE_ATHENA','.git','node_modules'} and not n.startswith('.')]
        for name in names:
            if Path(name).match(pattern):
                p=Path(directory)/name
                try:
                    if p.is_file():hits.append(p)
                except (FileNotFoundError,OSError):pass
    def key(p):
        try:return p.stat().st_mtime
        except (FileNotFoundError,OSError):return -1
    return sorted(hits, key=key,reverse=True)

def matched_file(folder,pattern,sha=None):
    for p in find_files(folder,pattern):
        if p.parent!=folder:continue
        try:
            if sha is None or digest(p)==sha:return p
        except (OSError,FileNotFoundError):pass
    return None

phase30=None; p30_rejections=[]
for p in find_files(ROOT,'ICHI2027_Phase30_SHAREABLE*.json'):
    try:
        rec=json.loads(p.read_text('utf-8'))
        if rec.get('phase')!='30_deterministic_synthetic_lineage_and_private_athena_binding_gate':continue
        if rec.get('synthetic_lineage',{}).get('synthetic_integrity_tests_passing')!=9 or not rec.get('gates',{}).get('deterministic_synthetic_lineage_verified'):continue
        st=matched_file(p.parent,'ICHI2027_Phase30_BINDING_VALIDATION_STATUS*.csv')
        it=matched_file(p.parent,'ICHI2027_Phase30_LINEAGE_INTEGRITY_TESTS*.csv')
        if st and it:phase30=(p,rec,st,it);break
        p30_rejections.append(f'{p.name}: missing Phase 30 CSV companions')
    except (OSError,ValueError,KeyError) as exc:p30_rejections.append(f'{p.name}: {exc}')
if not phase30:
    raise FileNotFoundError('No complete Phase 30 run found. '+str(p30_rejections[:5]))
P30_JSON,p30,P30_BIND,P30_TESTS=phase30
OUTPUT_DIR=P30_JSON.parent
bstatus=pd.read_csv(P30_BIND,dtype=str,keep_default_na=False)
tests30=pd.read_csv(P30_TESTS,dtype=str,keep_default_na=False)
assert len(bstatus)==19 and bstatus.binding_key.is_unique
assert len(tests30)==9 and tests30.passed.str.lower().eq('true').all()
assert p30['vocabulary']['reviewed_snapshot_verified_test_binding_count']==0 or p30['vocabulary']['local_frozen_snapshot_provided']

required29={
 'bundle':'ICHI2027_Phase29_SYNTHETIC_FHIR_WITH_PROCEDURES*.json',
 'lineage':'ICHI2027_Phase29_SYNTHETIC_LINEAGE_SIDECAR*.csv',
 'gaps':'ICHI2027_Phase29_GAP_DISPOSITION*.csv',
 'bindings':'ICHI2027_Phase29_BINDING_AUDIT*.csv',
 'required':'ICHI2027_Phase29_TARGET_REQUIRED_FIELD_AUDIT*.csv',
}
phase29=None
for jp in find_files(ROOT,'ICHI2027_Phase29_SHAREABLE*.json'):
    try:
        if digest(jp)!=p30['phase29_sha256']:continue
        rec=json.loads(jp.read_text('utf-8'))
        paired={key:matched_file(jp.parent,pat,p30['phase29_bundle_sha256'] if key=='bundle' else None)
                for key,pat in required29.items()}
        if all(paired.values()):phase29=(jp,rec,paired);break
    except (OSError,ValueError,KeyError):continue
if not phase29:
    raise FileNotFoundError('Phase 29 exact matching run/companion files not found; re-run Phase 30 in the correct folder or place original Phase 29 outputs beside it.')
P29_JSON,p29,P29=phase29
assert p29.get('pinned_imagewg_commit')==p30.get('pinned_imagewg_commit')
assert p29.get('pinned_omop55_commit')==p30.get('pinned_omop55_commit')
assert p29['original_48_gap_disposition']['remaining_blocked']==36
bundle=json.loads(P29['bundle'].read_text('utf-8'))
lineage=pd.read_csv(P29['lineage'],dtype=str,keep_default_na=False)
gaps=pd.read_csv(P29['gaps'],dtype=str,keep_default_na=False)
binding_audit=pd.read_csv(P29['bindings'],dtype=str,keep_default_na=False)
required=pd.read_csv(P29['required'],dtype=str,keep_default_na=False)
assert (len(lineage),len(gaps),len(binding_audit))==(16,48,19)
assert set(bstatus.binding_key)==set(binding_audit.binding_key)
print('Verified Phase 30:',P30_JSON.name)
print('Matched Phase 29 SHA-256:',digest(P29_JSON))
print('Destination:',OUTPUT_DIR)

Verified Phase 30: ICHI2027_Phase30_SHAREABLE.json
Matched Phase 29 SHA-256: 4526db7e341b2e474f1df931f411f1e8c8daf56437b1a70b1f789a8072c6abc2
Destination: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs


## Structural shadow gate — independent of vocabulary

The shadow representation uses **source-derived synthetic keys** and no concept IDs. It tests key relationships and repeat-import determinism only. The cohort check below is an **MRI evidence pairing fixture** (bilateral hippocampal baseline and follow-up); it does *not* establish MCI status or OMOP target-cohort equivalence.

In [3]:
# 3. Build deterministic source-derived shadow tables, independent of Phase 29 staging row order.
resources=[e['resource'] for e in bundle.get('entry',[])]
allrefs=[f"{r['resourceType']}/{r['id']}" for r in resources]
assert len(allrefs)==len(set(allrefs)), 'Duplicate FHIR resource references'
idx={f"{r['resourceType']}/{r['id']}":r for r in resources}
patients=sorted((r for r in resources if r.get('resourceType')=='Patient'), key=lambda r:r['id'])
studies=sorted((r for r in resources if r.get('resourceType')=='ImagingStudy'), key=lambda r:r['id'])
procs=sorted((r for r in resources if r.get('resourceType')=='Procedure'), key=lambda r:r['id'])
obs=sorted((r for r in resources if r.get('resourceType')=='Observation'), key=lambda r:r['id'])
assert (len(patients),len(studies),len(procs),len(obs))==(2,4,4,16)

# Critical source-semantics guard: unknown demographic fields cannot justify synthetic F/M values.
patient_semantics=pd.DataFrame([{
    'synthetic_patient_key':p['id'],
    'FHIR_gender_recorded':str(bool(p.get('gender'))),
    'FHIR_race_recorded':str(bool(p.get('extension') and any('race' in x.get('url','').lower() for x in p.get('extension',[])))),
    'FHIR_ethnicity_recorded':str(bool(p.get('extension') and any('ethnicity' in x.get('url','').lower() for x in p.get('extension',[])))),
    'gender_mapping_status':'SOURCE_NOT_ATTESTED' if not p.get('gender') else 'SOURCE_PRESENT_REVIEW_REQUIRED'
} for p in patients])


def ext(obs,tail,key):
    vals=[x.get(key) for x in obs.get('extension',[]) if x.get('url','').endswith('/'+tail)]
    return vals[0] if len(vals)==1 else None

def shadow_tables(b):
    rs=[e['resource'] for e in b['entry']]
    lookup={f"{r['resourceType']}/{r['id']}":r for r in rs}
    ps=sorted((r for r in rs if r['resourceType']=='Patient'),key=lambda r:r['id'])
    ss=sorted((r for r in rs if r['resourceType']=='ImagingStudy'),key=lambda r:r['id'])
    oo=sorted((r for r in rs if r['resourceType']=='Observation'),key=lambda r:r['id'])
    pmap={r['id']:i for i,r in enumerate(ps,1)}
    smap={r['id']:i for i,r in enumerate(ss,1)}
    person=[{'person_key':i,'patient_id':r['id']} for r,i in ((r,pmap[r['id']]) for r in ps)]
    proc_rows=[];imaging=[];features=[];measures=[];sidecar=[]
    for s in ss:
        pid=s['subject']['reference'].split('/')[-1]
        sr=s.get('procedureReference',{}).get('reference')
        procedure=lookup.get(sr,{})
        puid=[x['value'] for x in s.get('identifier',[]) if x.get('system')=='urn:dicom:uid']
        series=[x.get('uid') for x in s.get('series',[])]
        assert len(puid)==len(series)==1 and procedure.get('subject',{}).get('reference')==f'Patient/{pid}'
        proc_rows.append({'procedure_key':smap[s['id']],'person_key':pmap[pid],
                          'procedure_fhir_ref':sr, 'date':procedure['performedDateTime'][:10]})
        imaging.append({'imaging_key':smap[s['id']],'person_key':pmap[pid],
                        'procedure_key':smap[s['id']],'study_fhir_id':s['id'],
                        'study_uid_synthetic':puid[0],'series_uid_synthetic':series[0],
                        'study_date':s['started'][:10]})
    for i,o in enumerate(oo,1):
        p=o['subject']['reference'].split('/')[-1]
        fr=o['focus'][0]['reference'];sid=fr.split('/')[-1]
        assert fr in lookup and lookup[fr]['subject']['reference']==f'Patient/{p}'
        anatomy=o['bodySite']['coding'][0]['code']
        assert re.fullmatch(r'SYN-(HIP|ENT)-(LEFT|RIGHT)',anatomy)
        q=o['valueQuantity']
        assert q['system']=='http://unitsofmeasure.org' and q['unit']=='mm3'
        features.append({'feature_key':i,'person_key':pmap[p],'imaging_key':smap[sid],
                         'observation_id':o['id'],'anatomy_and_side':anatomy,
                         'timepoint':ext(o,'timepoint-label','valueString'),
                         'volume_mm3':float(q['value'])})
        measures.append({'measurement_key':i,'person_key':pmap[p],
                         'source_observation_id':o['id'],'volume_mm3':float(q['value'])})
        sidecar.append({'feature_key':i,'measurement_key':i,'source_observation_id':o['id'],
                        'device_fhir_ref':o['device']['reference'],
                        'trace_state':ext(o,'trace-state','valueCode'),
                        'trace_rule':ext(o,'trace-rule-version','valueString')})
    return {n:pd.DataFrame(v) for n,v in [('person',person),('procedure',proc_rows),
              ('imaging',imaging),('feature',features),('measurement',measures),('sidecar',sidecar)]}

shadow=shadow_tables(bundle)
for name,df in shadow.items():print(name,len(df))

person 2
procedure 4
imaging 4
feature 16
measurement 16
sidecar 16


In [4]:
# 4. Independent structural tests including exact second-run idempotence (shadow only).
def table_hashes(tables):
    return {k:hashlib.sha256(df.to_json(orient='records').encode('utf-8')).hexdigest()
            for k,df in sorted(tables.items())}

results=[]
def test(name,passed,detail):results.append({'test':name,'passed':bool(passed),'detail':detail})
test('S0_exact_source_and_shadow_cardinality',
     [len(shadow[k]) for k in ['person','procedure','imaging','feature','measurement']]==[2,4,4,16,16],
     '2 synthetic persons; 4 procedures and studies; 16 features and measurements')
test('S1_unique_deterministic_keys',all(shadow[k][key].is_unique for k,key in
     [('person','person_key'),('procedure','procedure_key'),('imaging','imaging_key'),
      ('feature','feature_key'),('measurement','measurement_key')]), 'No duplicate synthetic keys')
test('S2_all_imaging_to_procedure_links',
     set(shadow['imaging'].procedure_key)==set(shadow['procedure'].procedure_key),
     '4/4 imaging records resolve their manufactured synthetic procedure fixtures')
test('S3_all_features_link_correct_person_and_study',
     all(r['person_key']==shadow['imaging'].set_index('imaging_key').loc[r['imaging_key'],'person_key']
         for r in shadow['feature'].to_dict('records')),
     'No cross-patient feature-to-image joins')
test('S4_source_observation_set_reconciles_phase29',
     set(shadow['feature'].observation_id)==set(lineage.source_observation_id) and
     shadow['feature'].observation_id.is_unique,
     '16/16 exact synthetic FHIR observation IDs recovered')
test('S5_feature_measurement_identity_recoverable',
     all(r['source_observation_id'] in set(shadow['feature'].observation_id) for r in shadow['sidecar'].to_dict('records'))
     and len(shadow['sidecar'])==16, '16 source-to-feature-to-measurement sidecar links')
test('S6_source_demographics_not_fabricated',
     all(x=='SOURCE_NOT_ATTESTED' for x in patient_semantics.gender_mapping_status),
     'No gender F/M asserted for synthetic FHIR patients lacking source gender')
# Source-only paired bilateral hippocampal evidence, NOT MCI and NOT OMOP cohort equivalence.
h=shadow['feature'][shadow['feature'].anatomy_and_side.str.contains('SYN-HIP-')]
source_pair=[]
for person_key,grp in h.groupby('person_key'):
    keys={(r['timepoint'],r['anatomy_and_side']) for r in grp.to_dict('records')}
    if all((t,f'SYN-HIP-{s}') in keys for t in ['baseline','12-month'] for s in ['LEFT','RIGHT']):
        source_pair.append(person_key)
test('S7_source_only_bilateral_pair_fixture',len(source_pair)==2,
     '2 synthetic persons have bilateral hippocampal observations at both fixture time points; MCI unknown')
test('S8_shadow_import_idempotence',table_hashes(shadow)==table_hashes(shadow_tables(bundle)),
     'Identical source gives identical deterministic structural rows')
test('S9_synthetic_UIDs_unique',
     shadow['imaging'].study_uid_synthetic.is_unique and shadow['imaging'].series_uid_synthetic.is_unique,
     'Synthetic UIDs are distinct; real DICOM UID provenance unverified')
structural_tests=pd.DataFrame(results)
assert structural_tests.passed.all(), structural_tests.loc[~structural_tests.passed].to_dict('records')
print(f'Shadow checks passed: {structural_tests.passed.sum()}/{len(structural_tests)}')
print('NOTE: No OMOP/MI-CDM target table has been populated; these are structural fixtures only.')

Shadow checks passed: 10/10
NOTE: No OMOP/MI-CDM target table has been populated; these are structural fixtures only.


## Private vocabulary inputs — never upload to public GitHub

Create or use the `_PRIVATE_ATHENA` folder automatically created beside your Phase 30 files. Place these files **only there**:

- `CONCEPT.csv` or `CONCEPT.csv.gz` from your authorized, frozen Athena vocabulary export. The notebook accepts tab or comma separated files and streams them in chunks.
- `SNAPSHOT_MANIFEST.json` with the date, SHA-256 and origin of that *specific* export. A template is created below; a missing manifest blocks approval even if `CONCEPT.csv` exists.
- `ICHI2027_Phase31_REVIEWED_BINDINGS.csv` — reviewer-selected IDs, matching vocabulary SHA-256, reviewer name, date and mapping evidence. A blank template is created below and is never overwritten.

`CONCEPT` candidate suggestions stay **inside** `_PRIVATE_ATHENA`; search suggestions are **not** validated bindings. A user-supplied reviewed CSV, not this notebook, must document why the concept matches the source meaning. The optional event-field binding requires independent documentation of the metadata link target.

In [5]:
# 5. Create private templates safely. Do not overwrite reviewer choices.
PRIVATE=OUTPUT_DIR/'_PRIVATE_ATHENA'
PRIVATE.mkdir(parents=True,exist_ok=True)
SPECS={
 'IO_MODALITY_MR':('Imaging modality','magnetic resonance',''),
 'IF_HIP_LEFT':('MRI feature concept, left hippocampus','hippocamp',''),
 'IF_HIP_RIGHT':('MRI feature concept, right hippocampus','hippocamp',''),
 'IF_ENT_LEFT':('MRI feature concept, left entorhinal','entorhin',''),
 'IF_ENT_RIGHT':('MRI feature concept, right entorhinal','entorhin',''),
 'IF_TYPE_ALGORITHM':('Image feature generation method','algorithm',''),
 'PERSON_GENDER_F':('Female synthetic source sex (currently unrecorded)','female','Gender'),
 'PERSON_GENDER_M':('Male synthetic source sex (currently unrecorded)','male','Gender'),
 'PERSON_RACE_UNRECORDED':('Missing source race policy (not automatically assigned)','unknown','Race'),
 'PERSON_ETHNICITY_UNRECORDED':('Missing source ethnicity policy (not automatically assigned)','unknown','Ethnicity'),
 'PROCEDURE_MRI':('Synthetic MRI acquisition procedure','brain magnetic resonance','Procedure'),
 'PROCEDURE_TYPE_SYN':('Source type for manufactured procedure test','source','Type Concept'),
 'MEAS_HIP_LEFT':('Volume measurement, left hippocampus','hippocamp','Measurement'),
 'MEAS_HIP_RIGHT':('Volume measurement, right hippocampus','hippocamp','Measurement'),
 'MEAS_ENT_LEFT':('Volume measurement, left entorhinal','entorhin','Measurement'),
 'MEAS_ENT_RIGHT':('Volume measurement, right entorhinal','entorhin','Measurement'),
 'MEAS_TYPE_SYN':('Measurement extraction method','derived','Type Concept'),
 'UNIT_MM3':('Volume unit mm3','cubic millimeter','Unit'),
 'IF_EVENT_MEAS_ID':('Metadata reference for MEASUREMENT.measurement_id','measurement_id','Metadata'),
}
assert set(SPECS)==set(binding_audit.binding_key)
review_template=pd.DataFrame([{
    'binding_key':r['binding_key'],'target_field':r['target_field'],
    'synthetic_source_code':r['synthetic_source_code'],
    'semantic_question_for_reviewer':SPECS[r['binding_key']][0],
    'search_hint_ONLY_NOT_A_MAPPING':SPECS[r['binding_key']][1],
    'expected_domain_if_known':SPECS[r['binding_key']][2],
    'chosen_concept_id':'','reviewed_domain_id':'',
    'reviewed_snapshot_sha256':'',
    'reviewer':'','review_date_YYYY_MM_DD':'','mapping_evidence':'',
    'source_semantics_evidence':'',
    'extension_domain_policy_evidence':'','laterality_preservation_evidence':'',
    'event_field_cdm_table_field':'','event_field_definition_evidence':'',
    'manual_disposition':'REVIEW_PENDING'
} for r in binding_audit.to_dict('records')])
REVIEW=PRIVATE/'ICHI2027_Phase31_REVIEWED_BINDINGS.csv'
if not REVIEW.is_file():
    # Existing Phase30 reviewed CSV is not silently imported: different column names need explicit review.
    review_template.to_csv(REVIEW,index=False)
MANIFEST=PRIVATE/'SNAPSHOT_MANIFEST.json'
MANIFEST_TEMPLATE=PRIVATE/'SNAPSHOT_MANIFEST_TEMPLATE.json'
if not MANIFEST_TEMPLATE.is_file():
    MANIFEST_TEMPLATE.write_text(json.dumps({
        'snapshot_date_utc':'YYYY-MM-DD',
        'expected_CONCEPT_sha256':'PASTE_SHA256_OF_YOUR_PRIVATE_CONCEPT_FILE',
        'source':'Name/date of authorized Athena vocabulary export',
        'omop_vocabulary_release':'Fill in exact vocabulary release/version if available',
        'review_note':'Keep this metadata and CONCEPT.csv only in _PRIVATE_ATHENA'
    },indent=2),encoding='utf-8')
print('Private reviewer template:',REVIEW)
print('Private snapshot manifest template:',MANIFEST_TEMPLATE)
print('Do not put _PRIVATE_ATHENA or raw CONCEPT.csv on public GitHub.')

Private reviewer template: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/_PRIVATE_ATHENA/ICHI2027_Phase31_REVIEWED_BINDINGS.csv
Private snapshot manifest template: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase24_outputs/_PRIVATE_ATHENA/SNAPSHOT_MANIFEST_TEMPLATE.json
Do not put _PRIVATE_ATHENA or raw CONCEPT.csv on public GitHub.


In [6]:
# 6. Optional frozen vocabulary scan. No auto-selection; no raw candidate exports to public output.
CONCEPT=next((PRIVATE/n for n in ['CONCEPT.csv','CONCEPT.csv.gz','concept.csv','concept.csv.gz']
              if (PRIVATE/n).is_file()),None)
snapshot_hash=digest(CONCEPT) if CONCEPT else None
print('Private CONCEPT SHA-256 (copy into manifest and each reviewed decision):',snapshot_hash or 'No CONCEPT.csv supplied')
manifest={};snapshot_day=None;manifest_valid=False
if MANIFEST.is_file():
    try:
        manifest=json.loads(MANIFEST.read_text('utf-8'))
        snapshot_day=date.fromisoformat(str(manifest['snapshot_date_utc']))
        manifest_valid=(bool(manifest.get('source','').strip()) and
                        snapshot_day<=datetime.now(timezone.utc).date() and
                        bool(CONCEPT) and
                        str(manifest.get('expected_CONCEPT_sha256','')).strip().lower()==snapshot_hash)
    except (KeyError,ValueError,TypeError):
        print('Invalid SNAPSHOT_MANIFEST.json: use YYYY-MM-DD and an actual source.')

reviewed=pd.read_csv(REVIEW,dtype=str,keep_default_na=False)
required_review=set(review_template.columns)
if not required_review.issubset(reviewed.columns):
    raise ValueError('Phase31 reviewer CSV is missing columns: '+str(sorted(required_review-set(reviewed.columns))))
if reviewed.binding_key.duplicated().any() or set(reviewed.binding_key)!=set(review_template.binding_key):
    raise ValueError('Reviewed CSV must contain exactly 19 unique known binding keys.')
review_rows={r['binding_key']:r for r in reviewed.to_dict('records')}
selected={str(r['chosen_concept_id']).strip() for r in review_rows.values() if str(r['chosen_concept_id']).strip()}
if any(not x.isdigit() or int(x)<=0 for x in selected):
    raise ValueError('Chosen concept IDs must be positive integers. Never use placeholder IDs.')

matches={};candidates=[];candidate_counts={k:0 for k in SPECS}
needed=['concept_id','concept_name','domain_id','standard_concept',
        'valid_start_date','valid_end_date','invalid_reason']
if CONCEPT:
    op=gzip.open if CONCEPT.suffix=='.gz' else open
    with op(CONCEPT,'rt',encoding='utf-8-sig',errors='replace') as f:first=f.readline()
    sep='\t' if '\t' in first else ','
    # Python source has escaped tabs because this file is generated from a build script.
    if sep=='\t':sep='	'
    with op(CONCEPT,'rt',encoding='utf-8-sig',errors='replace') as f:
        header=next(csv.reader(f,delimiter=sep))
    if not set(needed).issubset(set(header)):
        raise ValueError('CONCEPT.csv is missing columns: '+str(sorted(set(needed)-set(header))))
    for chunk in pd.read_csv(CONCEPT,sep=sep,compression='infer',dtype=str,
                             keep_default_na=False,chunksize=50000,usecols=needed):
        for cr in chunk.loc[chunk.concept_id.isin(selected)].to_dict('records'):
            matches[cr['concept_id']]=cr
        # Private search hints only; max 8 per key and no automatic clinical mapping.
        current=chunk.loc[chunk.invalid_reason.eq('')]
        for k,(_,term,domain) in SPECS.items():
            if candidate_counts[k]>=8:continue
            subset=current.loc[current.concept_name.str.contains(term,case=False,regex=False,na=False)]
            if domain:subset=subset.loc[subset.domain_id.eq(domain)]
            elif k!='IF_EVENT_MEAS_ID':subset=subset.loc[subset.standard_concept.eq('S')]
            else:subset=subset.loc[subset.domain_id.eq('Metadata')]
            n=8-candidate_counts[k]
            for cr in subset.head(n).to_dict('records'):
                candidates.append({'binding_key':k,**cr,'status':'SEARCH_HINT_ONLY_NOT_A_MAPPING'})
                candidate_counts[k]+=1
    pd.DataFrame(candidates,columns=['binding_key']+needed+['status']).to_csv(
        PRIVATE/'ICHI2027_Phase31_PRIVATE_CANDIDATES_REVIEW_ONLY.csv',index=False)
print('Frozen vocabulary file supplied:',bool(CONCEPT))
print('Manifest supplied and valid:',manifest_valid)
print('Human-reviewed IDs supplied:',len(selected))
print('Candidate rows private only:',len(candidates))

Private CONCEPT SHA-256 (copy into manifest and each reviewed decision): No CONCEPT.csv supplied
Frozen vocabulary file supplied: False
Manifest supplied and valid: False
Human-reviewed IDs supplied: 0
Candidate rows private only: 0


In [7]:
# 7. Independently validate each reviewed selection without asserting it is clinically correct.
def parsed_date(raw):
    s=str(raw or '').strip()
    for fmt in ['%Y%m%d','%Y-%m-%d']:
        try:return datetime.strptime(s,fmt).date()
        except ValueError:continue
    return None

statuses=[];verified=set();today=datetime.now(timezone.utc).date()
for spec in review_template.to_dict('records'):
    k=spec['binding_key'];r=review_rows[k]
    cid=str(r['chosen_concept_id']).strip()
    status='BLOCKED_NO_REVIEWED_SELECTION'
    if cid:
        if not CONCEPT: status='BLOCKED_NO_FROZEN_CONCEPT_CSV'
        elif not manifest_valid:status='BLOCKED_NO_VERIFIABLE_SNAPSHOT_DATE_AND_SOURCE'
        elif str(r['reviewed_snapshot_sha256']).strip().lower()!=snapshot_hash:
            status='BLOCKED_REVIEW_REFERENCES_DIFFERENT_VOCABULARY_SNAPSHOT'
        elif not all(str(r[x]).strip() for x in ['reviewer','review_date_YYYY_MM_DD','mapping_evidence','source_semantics_evidence']):
            status='BLOCKED_INCOMPLETE_HUMAN_AND_SOURCE_REVIEW'
        elif not parsed_date(r['review_date_YYYY_MM_DD']) or parsed_date(r['review_date_YYYY_MM_DD'])>today:
            status='BLOCKED_INVALID_REVIEW_DATE'
        elif cid not in matches:status='BLOCKED_CONCEPT_NOT_FOUND_IN_SNAPSHOT'
        else:
            c=matches[cid]
            start=parsed_date(c['valid_start_date']);end=parsed_date(c['valid_end_date'])
            expected=spec['expected_domain_if_known']
            if c['invalid_reason'] or not start or not end or not(start<=snapshot_day<=end):
                status='BLOCKED_INVALID_OR_EXPIRED_CONCEPT'
            elif expected and c['domain_id']!=expected:status='BLOCKED_DOMAIN_MISMATCH'
            elif r['reviewed_domain_id']!=c['domain_id']:status='BLOCKED_REVIEWED_DOMAIN_MISMATCH'
            elif k!='IF_EVENT_MEAS_ID' and c['standard_concept']!='S':status='BLOCKED_NOT_STANDARD_CONCEPT'
            elif not expected and k!='IF_EVENT_MEAS_ID' and not str(r['extension_domain_policy_evidence']).strip():
                status='BLOCKED_UNDOCUMENTED_EXTENSION_DOMAIN_POLICY'
            elif k in {'IF_HIP_LEFT','IF_HIP_RIGHT','IF_ENT_LEFT','IF_ENT_RIGHT'} and not str(r['laterality_preservation_evidence']).strip():
                status='BLOCKED_UNDOCUMENTED_LATERALITY_PRESERVATION'
            elif k=='IF_EVENT_MEAS_ID' and not(
                c['domain_id']=='Metadata' and
                c['concept_name'].strip().lower()=='measurement_id' and
                str(r['event_field_cdm_table_field']).strip().upper()=='MEASUREMENT.MEASUREMENT_ID' and
                str(r['event_field_definition_evidence']).strip()):
                status='BLOCKED_EVENT_FIELD_METADATA_EVIDENCE_MISSING'
            elif k in {'PERSON_GENDER_F','PERSON_GENDER_M'} and not patient_semantics.FHIR_gender_recorded.eq('True').any():
                status='BLOCKED_SOURCE_GENDER_NOT_PRESENT_EVEN_IF_CONCEPT_VALID'
            elif k in {'PERSON_RACE_UNRECORDED','PERSON_ETHNICITY_UNRECORDED'} and not str(r['source_semantics_evidence']).strip():
                status='BLOCKED_MISSINGNESS_POLICY_NOT_DOCUMENTED'
            else:
                status='REVIEWED_AND_PRESENT_IN_FROZEN_SNAPSHOT_TEST_ONLY'
                verified.add(k)
    statuses.append({'binding_key':k,'target_field':spec['target_field'],
                     'verification_status':status,'snapshot_present':bool(CONCEPT),
                     'manifest_valid':manifest_valid,'review_selection_supplied':bool(cid),
                     'reviewed_snapshot_match':k in verified})
validation=pd.DataFrame(statuses)
print('Reviewed, snapshot-matched, source-supported binding keys:',len(verified),'of 19')
print('No concept IDs or raw/private Athena data will be exported in the public CSV.')

Reviewed, snapshot-matched, source-supported binding keys: 0 of 19
No concept IDs or raw/private Athena data will be exported in the public CSV.


In [8]:
# 8. Exact original-48 gap accounting + active mapping readiness.
by_id=shadow['feature'].set_index('feature_key')
original=gaps.copy(deep=True)
key_map={('HIP','LEFT'):'IF_HIP_LEFT',('HIP','RIGHT'):'IF_HIP_RIGHT',
         ('ENT','LEFT'):'IF_ENT_LEFT',('ENT','RIGHT'):'IF_ENT_RIGHT'}
keys=[];state=[]
for row in original.to_dict('records'):
    target=f"{row['table']}.{row['required_field']}"
    key=''
    if target=='image_occurrence.modality_concept_id':key='IO_MODALITY_MR'
    elif target=='image_feature.image_feature_type_concept_id':key='IF_TYPE_ALGORITHM'
    elif target=='image_feature.image_feature_concept_id':
        anatomical=by_id.loc[int(row['staging_row_id']),'anatomy_and_side'].split('-')
        key=key_map[(anatomical[1],anatomical[2])]
    if row['resolution_status']=='BLOCKED_VOCABULARY_VALIDATION' and not key:
        raise AssertionError('Unmapped required vocabulary gap encountered: '+target)
    keys.append(key)
    if row['resolution_status']!='BLOCKED_VOCABULARY_VALIDATION':
        state.append(row['resolution_status'])
    elif key in verified:
        state.append('REVIEWED_BINDING_READY_NOT_APPLIED_TO_TARGET')
    else:state.append('BLOCKED_VOCABULARY_VALIDATION')
original['phase31_binding_key']=keys
original['phase31_disposition']=state
assert len(original)==48
blocked=int(original.phase31_disposition.eq('BLOCKED_VOCABULARY_VALIDATION').sum())
ready=int(original.phase31_disposition.eq('REVIEWED_BINDING_READY_NOT_APPLIED_TO_TARGET').sum())
assert blocked+ready==36
assert original.phase31_disposition.eq('SYNTHETIC_FIXTURE_RESOLVED').sum()==8
assert original.phase31_disposition.eq('SYNTHETIC_PROCEDURE_FIXTURE_ONLY').sum()==4
print('Original 48: 8 UID test-only; 4 procedure test-only;',blocked,'vocabulary blocked;',ready,'reviewed-ready (NOT applied).')

Original 48: 8 UID test-only; 4 procedure test-only; 36 vocabulary blocked; 0 reviewed-ready (NOT applied).


In [9]:
# 9. Save all PUBLIC-SAFE outputs to the verified Phase30 Drive folder; private data stays private.
OUTPUTS={
 'shareable':OUTPUT_DIR/'ICHI2027_Phase31_SHAREABLE.json',
 'structural':OUTPUT_DIR/'ICHI2027_Phase31_SHADOW_STRUCTURE_TESTS.csv',
 'status':OUTPUT_DIR/'ICHI2027_Phase31_BINDING_REVIEW_STATUS.csv',
 'gap':OUTPUT_DIR/'ICHI2027_Phase31_GAP_DISPOSITION.csv',
 'source_semantics':OUTPUT_DIR/'ICHI2027_Phase31_SOURCE_SEMANTICS_AUDIT.csv',
}
structural_tests.to_csv(OUTPUTS['structural'],index=False)
validation.to_csv(OUTPUTS['status'],index=False)
original.to_csv(OUTPUTS['gap'],index=False)
patient_semantics.to_csv(OUTPUTS['source_semantics'],index=False)
summary={
 'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
 'phase':'31_frozen_athena_intake_source_semantics_and_synthetic_shadow_gate',
 'run_utc':datetime.now(timezone.utc).isoformat(),
 'privacy':'Only synthetic/public-safe statuses, no selected concept IDs, reviewer names, private Athena rows or ADNI records.',
 'phase30_sha256':digest(P30_JSON),'phase29_sha256':digest(P29_JSON),
 'pinned_imagewg_commit':p30['pinned_imagewg_commit'],
 'pinned_omop55_commit':p30['pinned_omop55_commit'],
 'structural_shadow':{
    'synthetic_person_count':len(shadow['person']),
    'synthetic_procedure_fixture_count':len(shadow['procedure']),
    'synthetic_image_occurrence_count':len(shadow['imaging']),
    'synthetic_image_feature_count':len(shadow['feature']),
    'synthetic_measurement_count':len(shadow['measurement']),
    'structural_checks_passing':int(structural_tests.passed.sum()),
    'structural_checks_total':len(structural_tests),
    'source_only_bilateral_hippocampal_pair_person_count':len(source_pair),
    'source_MCI_status_attested':False,
    'official_target_cohort_equivalence_tested':False,
    'idempotence_test_is_shadow_only':True,
 },
 'private_vocab_intake':{
    'frozen_CONCEPT_supplied':bool(CONCEPT),
    'private_CONCEPT_sha256':snapshot_hash,
    'snapshot_manifest_supplied_and_valid':manifest_valid,
    'reviewed_positive_IDs_supplied':len(selected),
    'snapshot_and_source_checked_keys':len(verified),
    'total_original_binding_keys':len(review_template),
    'private_candidate_suggestions_generated':len(candidates),
    'concept_ids_auto_selected':0,
    'athena_approval_claimed':False
 },
 'source_semantics':{
    'source_patient_gender_present_count':int(patient_semantics.FHIR_gender_recorded.eq('True').sum()),
    'gender_F_M_bindings_may_not_be_applied_without_source_evidence':True,
    'race_ethnicity_missingness_policy_required':True,
 },
 'original_48_gap_disposition':{
    'synthetic_UID_fixture_only_resolved':8,
    'synthetic_procedure_fixture_only_resolved':4,
    'vocabulary_review_ready_NOT_APPLIED':ready,
    'vocabulary_still_blocked':blocked,
    'original_phase29_missing_required_target_instances_not_silently_changed':82,
 },
 'gates':{
    'phase30_verified':True,
    'synthetic_shadow_structural_tests_passed':bool(structural_tests.passed.all()),
    'frozen_private_vocab_provided':bool(CONCEPT),
    'snapshot_manifest_verified':manifest_valid,
    'all_19_bindings_snapshot_and_source_checked':len(verified)==19,
    'all_36_original_vocab_gaps_reviewed_ready_not_applied':ready==36,
    'source_gender_attested':bool(patient_semantics.FHIR_gender_recorded.eq('True').all()),
    'synthetic_faithful_armA_completed':False,
    'pyOMOP_ETL_executed':False,
    'official_MICDM_load_completed':False,
    'real_MRI_rows_processed':0,
    'true_four_arm_comparison_completed':False,
    'original_generic_armC_executed':False,
    'OMOP_target_cohort_equivalence_tested':False,
 },
 'next_stage_gate':(
    'Supply an authorized frozen private CONCEPT.csv and a manifest; independently review 19 binding keys, '
    'document any absent-source demographic semantics and exact metadata event-field concept. '
    'Only after verified, source-supported concepts are available build a faithful synthetic Arm A target mapping '
    'and test required fields, exact linkage, SQL replay/idempotence and cohort equivalence. No synthetic-to-real extrapolation.'
 )
}
OUTPUTS['shareable'].write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding='utf-8')
readback=json.loads(OUTPUTS['shareable'].read_text('utf-8'))
assert readback['structural_shadow']['structural_checks_passing']==10
assert len(pd.read_csv(OUTPUTS['gap']))==48
assert readback['gates']['pyOMOP_ETL_executed'] is False
assert readback['gates']['official_MICDM_load_completed'] is False
assert readback['gates']['original_generic_armC_executed'] is False
print('PHASE 31: STRUCTURAL SHADOW CHECKS PASS; VOCABULARY GATE:',len(verified),'/19 source-and-snapshot-checked')
print('PUBLIC outputs saved to Drive:')
for p in OUTPUTS.values():print(' -',p.name)
print('PRIVATE review inputs remain in:',PRIVATE.name)

PHASE 31: STRUCTURAL SHADOW CHECKS PASS; VOCABULARY GATE: 0 /19 source-and-snapshot-checked
PUBLIC outputs saved to Drive:
 - ICHI2027_Phase31_SHAREABLE.json
 - ICHI2027_Phase31_SHADOW_STRUCTURE_TESTS.csv
 - ICHI2027_Phase31_BINDING_REVIEW_STATUS.csv
 - ICHI2027_Phase31_GAP_DISPOSITION.csv
 - ICHI2027_Phase31_SOURCE_SEMANTICS_AUDIT.csv
PRIVATE review inputs remain in: _PRIVATE_ATHENA


## After running

Upload the following **public-safe** files here: `ICHI2027_Phase31_SHAREABLE.json`, `ICHI2027_Phase31_SHADOW_STRUCTURE_TESTS.csv` and `ICHI2027_Phase31_BINDING_REVIEW_STATUS.csv`. The `GAP_DISPOSITION.csv` is also helpful.

Do **not** upload `_PRIVATE_ATHENA`, the `CONCEPT.csv` snapshot or any reviewer names. If Athena is not yet available, you can still run the notebook and return its public-safe outputs. Phase 32's **faithful** mapping remains gated by frozen vocabulary evidence and documented source demographics; synthetic shadow tests are not a substitute.